In [2]:
import numpy as np
import pandas as pd
from pyhwr import GHiampDataManager, LSLDataManager
import os

In [ ]:
carpet = "sub-06/ses-01"

path = "../data/raw_file/" + carpet
filename_hdf5 = "sub-06_ses-01_task-ejecutada_run-05_eeg.hdf5"
filename_xdf = "sub-06_ses-01_task-ejecutada_run-05_eeg.xdf"

gmanager = GHiampDataManager(os.path.join(path, filename_hdf5), normalize_time=True)
gmanager.changeMarkersNames({1: "inicioSesión", 2: "trialTablet", 3: "penDown", 4: "trialLaptop"})
markers = gmanager.markers_info

In [4]:
# Abro el .xdf para obtener los marcadores y las letras en cada trial
lslmanager = LSLDataManager(os.path.join(path, filename_xdf))
letters = lslmanager["Laptop_Markers", "letter", :]

In [5]:
# Me quedo con las muestras de los marcadores de interés
start_sesion = markers['inicioSesión']
trial_tablet = markers['trialTablet']

# Genero mi array de muestras de marcadores
samples = np.concatenate([start_sesion, trial_tablet])

# Genero mi array de marcadores
trial_type = np.array(['inicioSesion'] + letters) #type: ignore

# Obtengo mi señal en forma (n_canales, n_muestras)
signal = gmanager.raw_data.T #type: ignore

In [6]:
# Verifico la forma de mi señal
print(f"Shape de señal: {signal.shape}")

# Verifico que los marcadores queden en el orden correcto
print(f"'inicioSesión' debe ir al inicio: {trial_type}")

# Verifico el largo de 'samples'
print(f"Shape de muestras de marcadores: {samples.shape}")

Shape de señal: (67, 1028544)
'inicioSesión' debe ir al inicio: ['inicioSesion' 'n' 's' 'u' 'd' 'l' 'r' 'o' 'e' 'a' 'm' 'e' 'l' 'd' 'r'
 'm' 's' 'o' 'u' 'n' 'a' 'm' 's' 'l' 'a' 'o' 'n' 'u' 'e' 'd' 'r' 'a' 'd'
 'r' 'm' 'n' 'e' 'l' 'o' 's' 'u' 'o' 'u' 'n' 's' 'l' 'a' 'd' 'm' 'e' 'r'
 'r' 'e' 'a' 'l' 'd' 'n' 'o' 's' 'm' 'u' 'o' 'l' 'a' 'u' 'm' 'd' 'n' 'r'
 'e' 's' 'a' 'u' 'n' 'l' 'm' 'o' 'd' 's' 'r' 'e']
Shape de muestras de marcadores: (81,)


In [8]:
# Genero un DataFrame para obtener un archivo .tsv
df = pd.DataFrame({
    'onset': samples,
    'duration': 0.0,
    'trial_type': trial_type
})

# Verifico que la tabla se genere correctamente
df.head()

,onset,duration,trial_type
0,4.628333,0.0,inicioSesion
1,18.036667,0.0,n
2,27.920000,0.0,s
3,37.986667,0.0,u
4,48.120000,0.0,d


In [ ]:
subject = "sub-06-run-05"

# Exportar con tabulación como separador
df.to_csv(f'..data/tsv_file/{subject}_events.tsv', sep='\t', index=False)

# Exportar la señal en archivo -npy
np.save(f"../data/npy_file/{subject}_signal.npy", signal)